<h2>Install huggingface-cli</h2>
python3.10 -m pip install -U "huggingface_hub[cli]"
python3.10 -m pip show huggingface_hub

In [4]:

#pip install accelerate diffusers torch -----> too slow in local
# from diffusers import StableDiffusionPipeline
# import torch

# pipe = StableDiffusionPipeline.from_pretrained(
#     "OFA-Sys/small-stable-diffusion-v0",
#     torch_dtype=torch.float16,
#     device_map="cpu"
# )
# #first run may be slow to download files from Hugging Face (~0.3–0.6 GB)

# image = pipe("A cute robot learning Python").images[0]
# image.save("output.png")    #take approx.  for 1 image 

In [15]:
import importlib
import utilities
import shutil
import sys
import os
importlib.reload(utilities)
from utilities import get_first_file, rename_files
import requests
import argparse

from dotenv import load_dotenv
load_dotenv(override=True) 

PAGE_ID = os.environ['FB_MATH_KIDS_PAGE_ID']
PAGE_ACCESS_TOKEN = os.environ['FB_MATH_KIDS_TOKEN']

FB_GRAPH_VERSION = 'v24.0'

GRAPH_URL = f"https://graph.facebook.com/{FB_GRAPH_VERSION}/"
VIDEO_GRAPH_URL = f"{GRAPH_URL}{PAGE_ID}/videos"

CHUNK_SIZE = 4 * 1024 * 1024  # 4MB chunks

In [12]:
from huggingface_hub import InferenceClient

HF_TOKEN = os.environ['HF_TOKEN']

def generate_img():
    client = InferenceClient(
        # provider="fal-ai",   # auto select an Inference
        api_key=HF_TOKEN
    )

    image = client.text_to_image(
        prompt="A futuristic Ho Chi Minh city, cyberpunk style, ultra detailed",
        model="Qwen/Qwen-Image"
    )

    image.save("output.png")
#
# generate_img()

In [16]:
GEMINI_API_KEY = os.environ['GEMINI_API_KEY']

In [17]:
from google import genai
from google.genai import types
from PIL import Image
import io

# 1. Initialize the Client
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Generate the Image
# We use 'imagen-3.0-generate-002' (the current high-quality standard)
response = client.models.generate_images(
    model="imagen-3.0-generate-002",
    prompt="A futuristic classroom where kids are learning from floating holograms, digital art style",
    config=types.GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio="1:1",
        # You can add safety_filter_level="BLOCK_ONLY_HIGH" for more creative freedom
    )
)

# 3. Process and Save
for generated_image in response.generated_images:
    # Convert the raw bytes into a viewable image
    image = Image.open(io.BytesIO(generated_image.image.image_bytes))
    image.save("learning_output.png")
    image.show()

print("Image saved successfully!")

ImportError: cannot import name 'genai' from 'google' (unknown location)

In [2]:
VIDEO_FOLDER = '/Users/sangdo/Downloads/math_games_video/output/fb_video/'   #contain mp4 files

DESCRIPTION = """
300+ games as PDF file in the first comment.
Math games for your kids at the spare time - Puzzle {index}

We introduce a range of various games:
Addition matrix
Hidden gems
Word search
Crossword numbers
Balance game
Find lines
Triangle sum
Balance fruit
Object coordination
Spy game
Detect shape
Bee house
"""

COMMENT = os.environ['COMMENT_CONTENT']

In [3]:
def get_post_id_from_video_id(video_id):
    url = f"{GRAPH_URL}{video_id}"
    
    params = {
        "fields": "post_id",
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json().get("post_id")  # e.g. "123456789_987654321"

In [4]:
def start_upload(file_size):
    params = {
        "upload_phase": "start",
        "file_size": file_size,
        "access_token": PAGE_ACCESS_TOKEN
    }

    r = requests.post(VIDEO_GRAPH_URL, data=params, timeout=60)
    r.raise_for_status()
    return r.json()


def transfer_chunks(upload_session_id, start_offset, video_file):
    while True:
        start = int(start_offset)
        video_file.seek(start)
        chunk = video_file.read(CHUNK_SIZE)

        files = {
            "video_file_chunk": chunk
        }

        data = {
            "upload_phase": "transfer",
            "upload_session_id": upload_session_id,
            "start_offset": start_offset,
            "access_token": PAGE_ACCESS_TOKEN
        }

        r = requests.post(VIDEO_GRAPH_URL, data=data, files=files, timeout=300)
        r.raise_for_status()
        result = r.json()

        start_offset = result["start_offset"]
        end_offset = result["end_offset"]

        print(f"Uploaded bytes {start_offset} / {end_offset}")

        if start_offset == end_offset:
            break

    return start_offset

#uploaded video, now creating post
def finish_upload(upload_session_id, index):
    description = DESCRIPTION.replace('{index}', str(index))
    data = {
        "upload_phase": "finish",
        "upload_session_id": upload_session_id,
        "description": description,
        "published": "true",
        "video_state": "PUBLISHED",
        "access_token": PAGE_ACCESS_TOKEN
    }

    r = requests.post(VIDEO_GRAPH_URL, data=data, timeout=60)
    r.raise_for_status()
    print('Result after posting to FB page: ', r.json())
    return r.json()

#this is using streaming upload
#FB won't auto convert it to Reel and show in timeline
def upload_video():
    video_path, index = get_first_file(VIDEO_FOLDER, 'mp4')

    file_size = os.path.getsize(video_path)

    print("Starting upload session...")
    start = start_upload(file_size)

    upload_session_id = start["upload_session_id"]
    start_offset = start["start_offset"]

    with open(video_path, "rb") as f:
        transfer_chunks(upload_session_id, start_offset, f)

    print("Finishing upload... " + str(index))
    finish = finish_upload(upload_session_id, index)
    #there is no video id returned

    # video_id = finish["video_id"]
    # print("Video uploaded successfully!")
    # print("Video ID:", video_id)
    # post_id = get_post_id(video_id)
    # print("Post ID:", post_id)
    # #rename uploaded file with different extension (for comment later)
    # os.rename(video_path, video_path.replace('.mp4', '.fb'))

    return finish

#
def upload_video_and_publish():
    video_path, index = get_first_file(VIDEO_FOLDER, 'mp4')
    files = {
        "source": open(video_path, "rb")
    }
    description = DESCRIPTION.replace('{index}', str(index))
    data = {
        "access_token": PAGE_ACCESS_TOKEN,
        "description": description,
        "published": "true",
        "video_state": "PUBLISHED"
    }
    response = requests.post(VIDEO_GRAPH_URL, files=files, data=data)
    return response.json()


# upload_result = upload_video_and_publish()
# print(upload_result)   #{'id': '1634040370953948'}  #post id, till not published in Web, only in mobile

In [5]:
def get_latest_video_id():
    params = {
        "fields": "id,post_id",
        "limit": 1,                    # Only get the latest one
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(VIDEO_GRAPH_URL, params=params, timeout=60)
    r.raise_for_status()
    
    videos = r.json().get("data", [])
    if not videos:
        raise Exception("No videos found")
    print(videos)
    
    return videos[0]["id"], videos[0].get("post_id")

# get_latest_video_id()

In [6]:
def get_video_detail(video_id):
    url = f"{GRAPH_URL}{video_id}"
    
    params = {
        "fields": "id,title,description,length,place,status,created_time,thumbnails,permalink_url",
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()
#
# video_detail = get_video_detail('1690219669060171')
# print(video_detail)

In [7]:
def delete_post(post_id):
    url = f"{GRAPH_URL}{post_id}"
    
    params = {
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.delete(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()  # returns {"success": true}
#
# delete_post('')

In [8]:
# post_id = get_post_id_from_video_id('1690219669060171')
# print("Post ID:", post_id)

In [9]:
def get_first_comment(post_id):
    url = f"{GRAPH_URL}{post_id}/comments"
    
    params = {
        "fields": "id,message,created_time,from",
        "limit": 1,                    # Only fetch the first comment
        "order": "chronological",      # Oldest first
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    
    data = r.json().get("data", [])
    if not data:
        return None  # No comments yet
    
    return data[0]
#
# first_comment = get_first_comment('') #only post from API can read comment
# print(first_comment)


In [10]:
def check_token_permissions():
    url = "{GRAPH_URL}me"
    
    params = {"access_token": PAGE_ACCESS_TOKEN}
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# Check what permissions your token actually has
# perms = check_token_permissions()
# print(perms)


In [11]:
def create_comment(post_id, message):
    url = f"{GRAPH_URL}{post_id}/comments"
    
    data = {
        "message": message,
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.post(url, data=data, timeout=60)
    r.raise_for_status()
    return r.json()

##
# comment_result = create_comment('', COMMENT)
# print('Commented result:', comment_result)